# Customer Churn Pattern Analysis

## Project Overview
This notebook analyzes customer-level telecom data to identify patterns associated with churn. The workflow focuses on churn distribution, feature relationships, risk-factor segmentation, and business actions that could reduce avoidable churn.

## Learning Objectives
- Understand how churn is distributed across the customer base.
- Compare churned and retained customers across demographic, contract, service, and billing features.
- Build risk-focused customer segments instead of relying on one-variable summaries.
- Translate descriptive churn analysis into practical retention recommendations.

## Business Problem
Customer churn directly reduces recurring revenue and increases acquisition pressure. A business needs to know which customer profiles are most at risk, which service or pricing patterns are associated with churn, and where retention efforts are most likely to pay off.

## Why This Analysis Matters
Retaining an existing customer is usually cheaper than replacing one. A clear view of churn patterns helps teams prioritize contract design, support interventions, pricing changes, and offer strategy more effectively.


## Dataset Overview
This notebook uses the repo-local telecom churn dataset at `data/customer_churn_telecom/telecom_customer_churn.csv`. The verified schema includes demographic fields, contract and service information, billing variables, revenue measures, and churn-related fields such as `Customer Status`, `Churn Category`, and `Churn Reason`.

## Dataset Notes
The `Customer Status` column has three states in this dataset: `Stayed`, `Churned`, and `Joined`. For churn pattern analysis, the notebook compares `Stayed` and `Churned` customers directly and keeps `Joined` visible in overall status reporting but out of the main churn-risk comparison to avoid mixing new customers with established retention outcomes.


## Environment Setup
The next cell installs only the packages needed for this notebook so the analysis can be rerun in a clean environment without hidden setup assumptions.


In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}

def ensure_package(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

for package_name, import_name in REQUIRED_PACKAGES.items():
    ensure_package(package_name, import_name)

print('Environment setup complete.')


## Imports
These imports support data loading, cleaning, summarization, and visualization. The notebook keeps its dependency surface intentionally small.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
np.random.seed(42)


## Configuration / Constants
This block locates the dataset, defines the expected schema, and makes the main notebook assumptions explicit. Keeping these in one place makes the analysis easier to review later.


In [ ]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'README.md').exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
DATASET_PATH = WORKSPACE_ROOT / 'data' / 'customer_churn_telecom' / 'telecom_customer_churn.csv'
EXPECTED_COLUMNS = [
    'Customer ID', 'Gender', 'Age', 'Married', 'Number of Dependents', 'City', 'Zip Code',
    'Latitude', 'Longitude', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service',
    'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type',
    'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan',
    'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music',
    'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge',
    'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges',
    'Total Revenue', 'Customer Status', 'Churn Category', 'Churn Reason'
]

config_preview = {
    'dataset_path': str(DATASET_PATH),
    'expected_column_count': len(EXPECTED_COLUMNS),
}
print(json.dumps(config_preview, indent=2))


## Dataset Loading
The dataset is loaded from the repo-local CSV file. If the file is missing, the notebook raises a clear path-specific error instead of failing later with a misleading message.


In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Expected dataset not found at: {DATASET_PATH}')

raw_df = pd.read_csv(DATASET_PATH)
print(f'Raw shape: {raw_df.shape}')
display(raw_df.head())


## Data Validation Checks
Before comparing churned and retained customers, this block checks the schema, missingness, status distribution, and duplicate customer identifiers. That keeps the rest of the analysis grounded in verified inputs.


In [ ]:
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(raw_df.columns))
extra_columns = sorted(set(raw_df.columns) - set(EXPECTED_COLUMNS))

validation_report = pd.Series({
    'row_count': int(len(raw_df)),
    'missing_columns': missing_columns,
    'extra_columns': extra_columns,
    'duplicate_customer_ids': int(raw_df.duplicated(subset=['Customer ID']).sum()),
    'unique_customer_ids': int(raw_df['Customer ID'].nunique()),
    'status_levels': raw_df['Customer Status'].value_counts(dropna=False).to_dict(),
}).to_frame(name='value')

missing_summary = raw_df.isna().sum().sort_values(ascending=False).to_frame(name='missing_count')

display(validation_report)
display(missing_summary.head(20))


## Data Cleaning And Churn Label Preparation
This step standardizes the main fields, converts numeric columns explicitly, and builds a clean churn-analysis subset. `Joined` customers remain visible in the overall status view, but the primary churn comparison uses only `Stayed` and `Churned` customers to avoid mixing retention outcomes with newly acquired customers.


In [ ]:
df = raw_df.copy()
df = df.drop_duplicates(subset=['Customer ID']).copy()

numeric_columns = [
    'Age', 'Number of Dependents', 'Number of Referrals', 'Tenure in Months',
    'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge',
    'Total Charges', 'Total Refunds', 'Total Extra Data Charges',
    'Total Long Distance Charges', 'Total Revenue'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

string_columns = [
    'Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service',
    'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan',
    'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music',
    'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method',
    'Customer Status', 'Churn Category', 'Churn Reason'
]
for column in string_columns:
    df[column] = df[column].astype(str).str.strip()
    df[column] = df[column].replace({'nan': np.nan})

df['Offer'] = df['Offer'].fillna('No Offer')
df['Churn Category'] = df['Churn Category'].fillna('Not Churned')
df['Churn Reason'] = df['Churn Reason'].fillna('Not Churned')

analysis_df = df[df['Customer Status'].isin(['Stayed', 'Churned'])].copy()
analysis_df['Churn Flag'] = (analysis_df['Customer Status'] == 'Churned').astype(int)
analysis_df['Tenure Band'] = pd.cut(analysis_df['Tenure in Months'], bins=[-1, 6, 12, 24, 48, 72], labels=['0-6', '7-12', '13-24', '25-48', '49-72'])
analysis_df['Age Band'] = pd.cut(analysis_df['Age'], bins=[17, 29, 39, 49, 59, 69, 100], labels=['18-29', '30-39', '40-49', '50-59', '60-69', '70+'])
analysis_df['Monthly Charge Band'] = pd.qcut(analysis_df['Monthly Charge'], q=4, duplicates='drop')

cleaning_report = pd.Series({
    'full_customer_rows': int(len(df)),
    'analysis_rows_stayed_or_churned': int(len(analysis_df)),
    'joined_customers_excluded_from_main_risk_analysis': int((df['Customer Status'] == 'Joined').sum()),
    'analysis_churn_rate_pct': float(analysis_df['Churn Flag'].mean() * 100),
}).to_frame(name='value')

display(cleaning_report)
display(analysis_df.head())


## Churn Distribution
This section shows how the customer base splits across status groups and how strong the churn rate is once the analysis is restricted to comparable customers. Starting here keeps the rest of the notebook anchored to the size of the actual problem.


In [ ]:
status_distribution = df['Customer Status'].value_counts(dropna=False).rename_axis('Customer Status').reset_index(name='customer_count')
churn_distribution = analysis_df['Customer Status'].value_counts().rename_axis('Customer Status').reset_index(name='customer_count')
overall_churn_rate = analysis_df['Churn Flag'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=status_distribution, x='Customer Status', y='customer_count', ax=axes[0], palette='Set2')
axes[0].set_title('Customer Status Distribution')

sns.barplot(data=churn_distribution, x='Customer Status', y='customer_count', ax=axes[1], palette='Set1')
axes[1].set_title(f'Stayed vs Churned Distribution | Churn Rate = {overall_churn_rate:.2f}%')

plt.tight_layout()
plt.show()

display(status_distribution)
display(churn_distribution)


## Feature Relationships: Numeric Variables
The next block compares churned and retained customers across continuous or count-like features. This helps identify whether churn tends to cluster around low tenure, high monthly charge, lower total revenue, or other measurable profiles.


In [ ]:
numeric_compare_columns = ['Age', 'Tenure in Months', 'Monthly Charge', 'Total Charges', 'Total Revenue', 'Number of Referrals', 'Avg Monthly GB Download']
numeric_summary = analysis_df.groupby('Customer Status')[numeric_compare_columns].mean().transpose()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.boxplot(data=analysis_df, x='Customer Status', y='Tenure in Months', ax=axes[0, 0], palette='coolwarm')
axes[0, 0].set_title('Tenure by Customer Status')

sns.boxplot(data=analysis_df, x='Customer Status', y='Monthly Charge', ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Monthly Charge by Customer Status')

sns.boxplot(data=analysis_df, x='Customer Status', y='Total Revenue', ax=axes[1, 0], palette='magma')
axes[1, 0].set_title('Total Revenue by Customer Status')

sns.scatterplot(data=analysis_df.sample(min(len(analysis_df), 3000), random_state=42), x='Tenure in Months', y='Monthly Charge', hue='Customer Status', alpha=0.4, ax=axes[1, 1])
axes[1, 1].set_title('Tenure vs Monthly Charge')

plt.tight_layout()
plt.show()

display(numeric_summary)


## Feature Relationships: Categorical Risk Factors
Churn is often driven by combinations of contract, internet service, support, payment, and billing behavior. This block calculates churn rates across several categorical dimensions so we can see where risk is concentrated.


In [ ]:
def churn_rate_table(frame, column_name):
    summary = frame.groupby(column_name, dropna=False).agg(
        customer_count=('Customer ID', 'count'),
        churn_rate=('Churn Flag', 'mean'),
        avg_monthly_charge=('Monthly Charge', 'mean'),
        avg_total_revenue=('Total Revenue', 'mean'),
    ).reset_index()
    summary['churn_rate_pct'] = summary['churn_rate'] * 100
    return summary.sort_values(['churn_rate_pct', 'customer_count'], ascending=[False, False])

contract_risk = churn_rate_table(analysis_df, 'Contract')
internet_type_risk = churn_rate_table(analysis_df, 'Internet Type')
payment_method_risk = churn_rate_table(analysis_df, 'Payment Method')
paperless_risk = churn_rate_table(analysis_df, 'Paperless Billing')
tech_support_risk = churn_rate_table(analysis_df, 'Premium Tech Support')
offer_risk = churn_rate_table(analysis_df, 'Offer')

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
sns.barplot(data=contract_risk, x='Contract', y='churn_rate_pct', ax=axes[0, 0], palette='rocket')
axes[0, 0].set_title('Churn Rate by Contract')

sns.barplot(data=internet_type_risk, x='Internet Type', y='churn_rate_pct', ax=axes[0, 1], palette='crest')
axes[0, 1].set_title('Churn Rate by Internet Type')
axes[0, 1].tick_params(axis='x', rotation=30)

sns.barplot(data=payment_method_risk, x='Payment Method', y='churn_rate_pct', ax=axes[0, 2], palette='flare')
axes[0, 2].set_title('Churn Rate by Payment Method')
axes[0, 2].tick_params(axis='x', rotation=30)

sns.barplot(data=paperless_risk, x='Paperless Billing', y='churn_rate_pct', ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Churn Rate by Paperless Billing')

sns.barplot(data=tech_support_risk, x='Premium Tech Support', y='churn_rate_pct', ax=axes[1, 1], palette='mako')
axes[1, 1].set_title('Churn Rate by Premium Tech Support')

sns.barplot(data=offer_risk, x='Offer', y='churn_rate_pct', ax=axes[1, 2], palette='Set2')
axes[1, 2].set_title('Churn Rate by Offer')
axes[1, 2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

display(contract_risk)
display(internet_type_risk)
display(payment_method_risk)


## Segmentation By Risk Factors
Single-variable views are useful, but churn management usually happens at the segment level. This section combines risk factors to show where churn is concentrated across tenure, age, contract, and internet-service patterns.


In [ ]:
tenure_band_risk = churn_rate_table(analysis_df, 'Tenure Band')
age_band_risk = churn_rate_table(analysis_df, 'Age Band')
contract_internet_heatmap = analysis_df.pivot_table(index='Contract', columns='Internet Type', values='Churn Flag', aggfunc='mean') * 100
tenure_contract_segment = analysis_df.groupby(['Tenure Band', 'Contract'], dropna=False).agg(
    customer_count=('Customer ID', 'count'),
    churn_rate=('Churn Flag', 'mean'),
    avg_monthly_charge=('Monthly Charge', 'mean'),
).reset_index()
tenure_contract_segment['churn_rate_pct'] = tenure_contract_segment['churn_rate'] * 100
high_risk_segments = tenure_contract_segment[tenure_contract_segment['customer_count'] >= 100].sort_values('churn_rate_pct', ascending=False).head(10)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=tenure_band_risk, x='Tenure Band', y='churn_rate_pct', ax=axes[0], palette='coolwarm')
axes[0].set_title('Churn Rate by Tenure Band')

sns.barplot(data=age_band_risk, x='Age Band', y='churn_rate_pct', ax=axes[1], palette='Spectral')
axes[1].set_title('Churn Rate by Age Band')

sns.heatmap(contract_internet_heatmap, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[2])
axes[2].set_title('Churn Rate % by Contract and Internet Type')

plt.tight_layout()
plt.show()

display(high_risk_segments)


## Revenue Exposure And Churn Reasons
Not all churn hurts equally. This section looks at revenue patterns and churn reasons together so the business can prioritize the segments that combine high risk with meaningful value at stake.


In [ ]:
revenue_by_status = analysis_df.groupby('Customer Status', as_index=False).agg(
    customer_count=('Customer ID', 'count'),
    total_revenue=('Total Revenue', 'sum'),
    avg_total_revenue=('Total Revenue', 'mean'),
    avg_monthly_charge=('Monthly Charge', 'mean'),
)

revenue_at_risk_by_contract = analysis_df.groupby('Contract', as_index=False).agg(
    customer_count=('Customer ID', 'count'),
    churn_rate=('Churn Flag', 'mean'),
    total_revenue=('Total Revenue', 'sum'),
    avg_revenue=('Total Revenue', 'mean'),
)
revenue_at_risk_by_contract['estimated_revenue_at_risk'] = revenue_at_risk_by_contract['churn_rate'] * revenue_at_risk_by_contract['total_revenue']
revenue_at_risk_by_contract = revenue_at_risk_by_contract.sort_values('estimated_revenue_at_risk', ascending=False)

churned_only = analysis_df[analysis_df['Churn Flag'] == 1].copy()
churn_category_summary = churned_only['Churn Category'].value_counts().rename_axis('Churn Category').reset_index(name='customer_count')
churn_reason_summary = churned_only['Churn Reason'].value_counts().rename_axis('Churn Reason').reset_index(name='customer_count').head(10)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=revenue_by_status, x='Customer Status', y='avg_total_revenue', ax=axes[0], palette='Set2')
axes[0].set_title('Average Total Revenue by Status')

sns.barplot(data=revenue_at_risk_by_contract, x='Contract', y='estimated_revenue_at_risk', ax=axes[1], palette='rocket')
axes[1].set_title('Estimated Revenue at Risk by Contract')

sns.barplot(data=churn_category_summary, x='Churn Category', y='customer_count', ax=axes[2], palette='magma')
axes[2].set_title('Churn Category Distribution')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

display(revenue_by_status)
display(revenue_at_risk_by_contract)
display(churn_category_summary)
display(churn_reason_summary)


## Actionable Business Insights
The next cell summarizes the main findings from the computed tables so the final takeaways stay grounded in the actual analysis results instead of prewritten claims.


In [ ]:
highest_risk_contract = contract_risk.iloc[0]
highest_risk_internet = internet_type_risk.iloc[0]
highest_risk_payment = payment_method_risk.iloc[0]
highest_risk_tenure = tenure_band_risk.iloc[0]
top_revenue_risk_contract = revenue_at_risk_by_contract.iloc[0]
top_churn_category = churn_category_summary.iloc[0]
top_churn_reason = churn_reason_summary.iloc[0]

insights = [
    f'Overall churn rate among comparable retained-vs-churned customers is {overall_churn_rate:.2f}%.',
    f'Highest-risk contract type: {highest_risk_contract["Contract"]} with churn rate {highest_risk_contract["churn_rate_pct"]:.2f}%.',
    f'Highest-risk internet type: {highest_risk_internet["Internet Type"]} with churn rate {highest_risk_internet["churn_rate_pct"]:.2f}%.',
    f'Highest-risk payment method: {highest_risk_payment["Payment Method"]} with churn rate {highest_risk_payment["churn_rate_pct"]:.2f}%.',
    f'Highest-risk tenure band: {highest_risk_tenure["Tenure Band"]} with churn rate {highest_risk_tenure["churn_rate_pct"]:.2f}%.',
    f'Largest estimated contract-level revenue exposure appears in {top_revenue_risk_contract["Contract"]}.',
    f'Top churn category among churned customers: {top_churn_category["Churn Category"]}.',
    f'Most frequent churn reason in the top-10 breakdown: {top_churn_reason["Churn Reason"]}.',
]

for item in insights:
    print(f'- {item}')


## Business Recommendations To Reduce Churn
- Prioritize retention for the highest-risk contract and tenure segments first, because these customers combine elevated churn probability with meaningful revenue exposure.
- Review offer strategy for early-tenure customers to reduce fast churn before habits become unstable.
- Target customers with weak support or higher-friction service profiles using proactive outreach, service checks, and product education.
- Reassess payment and billing friction, especially if the riskiest payment method and paperless-billing profile stay elevated after controlling for customer count.
- Use churn category and churn reason summaries to separate competitor-driven churn from dissatisfaction-driven churn, since those require different interventions.

## Limitations
- This is a descriptive analysis, not a causal model, so the notebook identifies associations rather than proven causes.
- Joined customers are intentionally excluded from the main churn-risk comparison, which improves comparability but narrows the scope of the analysis.
- Some operational context, such as campaign history or service outage logs, is not present in the dataset.

## Common Mistakes
- Treating all non-churned customers as equivalent when the dataset contains newly joined customers.
- Ranking risk segments by churn rate alone without checking customer count or revenue exposure.
- Confusing churn reasons with root causes when they may reflect only the final recorded explanation.

## Mini Challenge
1. Compare churn rates within the highest-risk contract type across age bands and tenure bands.
2. Build a simple churn-risk score using the top categorical and numeric signals discovered in this notebook.
3. Re-run the segment analysis using only customers above median total revenue to focus on high-value churn.

## Final Summary / Key Takeaways
This notebook builds a customer-level churn analysis workflow that starts with label hygiene, then moves through churn distribution, numeric and categorical feature relationships, risk segmentation, churn-reason breakdowns, and business recommendations. The core discipline is to analyze comparable customers, look at segment-level risk instead of single features only, and tie retention priorities to both churn probability and revenue exposure.
